# Laboratorio No. 1 Analizador Morfosintáctico con HMM
Presentado por:
VIVIANA ANDREA BAUTISTA PULIDO -
CARMEN EDILIA RICARDO GELVES -
ALEJANDRO DE MENDOZA

Este notebook del laboratorio de "Procesamiento del Lenguaje Natural" utiliza un modelo oculto de Markov (HMM) para analizar frases y etiquetarlas morfosintácticamente utilizando un corpus POS.

In [75]:
print("\n=== Punto 1: Cargar el Archivo del Corpus ===\n")

from google.colab import files
uploaded = files.upload()
corpus_file_path = list(uploaded.keys())[0]
print("Archivo cargado:", corpus_file_path)



=== Punto 1: Cargar el Archivo del Corpus ===



Saving corpus_etiquetado.txt to corpus_etiquetado (10).txt
Archivo cargado: corpus_etiquetado (10).txt


In [76]:
print("\n=== Punto 2: Procesar el Corpus ===\n")

def cargar_corpus(file_path):
    """Carga el corpus desde un archivo de texto, devolviendo una lista de oraciones (cada una como lista de tuplas (token, etiqueta)) con su doc_id."""
    oraciones = []
    corpus_flat = []
    oracion_actual = []
    doc_id = None

    with open(file_path, "r", encoding="utf-8") as file:
        for linea in file:
            linea = linea.strip()
            if linea.startswith("<doc"):
                # Finalizar la oración actual si existe
                if oracion_actual:
                    oraciones.append((doc_id, oracion_actual))
                    oracion_actual = []
                # Extraer el ID del documento
                doc_id = linea.split('id=')[1].strip('>') if 'id=' in linea else None
                continue
            if linea == "":
                # Finalizar la oración actual si existe
                if oracion_actual:
                    oraciones.append((doc_id, oracion_actual))
                    oracion_actual = []
                continue
            datos = linea.split("\t")
            if len(datos) >= 3:
                tupla = (datos[0], datos[2])
                oracion_actual.append(tupla)
                corpus_flat.append(tupla)

    # Añadir la última oración si no está vacía
    if oracion_actual:
        oraciones.append((doc_id, oracion_actual))

    return oraciones, corpus_flat

oraciones, corpus_flat = cargar_corpus(corpus_file_path)
print("Corpus cargado. Número de oraciones:", len(oraciones))
print("Número total de tokens:", len(corpus_flat))
print("\nOración 1 (doc id=1, completa):")
for doc_id, oracion in oraciones:
    if doc_id == "1":
        print(oracion)
        break
print("\nOración 2 (doc id=2, completa):")
for doc_id, oracion in oraciones:
    if doc_id == "2":
        print(oracion)
        break
print("\nResumen de primeras 3 oraciones (primeros 5 tokens):")
for doc_id, oracion in oraciones[:3]:
    print(f"Doc id={doc_id}: {oracion[:5]} {'...' if len(oracion) > 5 else ''}")


=== Punto 2: Procesar el Corpus ===

Corpus cargado. Número de oraciones: 20
Número total de tokens: 137

Oración 1 (doc id=1, completa):

Oración 2 (doc id=2, completa):

Resumen de primeras 3 oraciones (primeros 5 tokens):
Doc id="1": [('Habla', 'VMIP3S0'), ('con', 'SP'), ('el', 'DA'), ('enfermo', 'NCMS000'), ('grave', 'AQ0CS00')] ...
Doc id="2": [('El', 'DA'), ('enfermo', 'NCMS000'), ('grave', 'AQ0CS00'), ('habla', 'VMIP3S0'), ('de', 'SP')] ...
Doc id="3": [('La', 'DA'), ('película', 'NCFS000'), ('fue', 'VAIP3S0'), ('nominada', 'VMP00SF'), ('al', 'SP+DA')] ...


In [77]:
print("\n=== Punto 3: Implementación de Calcular Probabilidades con Suavizado de Laplace ===\n")

import numpy as np

def calcular_probabilidades_emision(corpus):
    """
    Calcula las probabilidades de emisión P(palabra|etiqueta) con suavizado de Laplace.
    Args:
        corpus (list): Lista de tuplas (token, etiqueta) del corpus completo.
    Returns:
        dict: Diccionario con etiquetas como claves y otro diccionario {palabra: probabilidad} como valor.
    """
    emision = {}
    total_etiquetas = {}
    palabras = set(token for token, _ in corpus)  # Vocabulario de palabras únicas
    k = 1  # Constante de suavizado de Laplace

    # Inicializar contadores para cada etiqueta
    for etiqueta in set(etiqueta for _, etiqueta in corpus):
        emision[etiqueta] = {}
        total_etiquetas[etiqueta] = 0

    # Contar frecuencias de palabra por etiqueta
    for token, etiqueta in corpus:
        emision[etiqueta][token] = emision[etiqueta].get(token, 0) + 1
        total_etiquetas[etiqueta] += 1

    # Aplicar suavizado de Laplace
    for etiqueta in emision:
        for palabra in palabras:
            # P(palabra|etiqueta) = (count(palabra, etiqueta) + k) / (total_count(etiqueta) + k * |vocab|)
            emision[etiqueta][palabra] = (emision[etiqueta].get(palabra, 0) + k) / (total_etiquetas[etiqueta] + k * len(palabras))

    # Calcular métricas de efectividad
    print("\n=== Métricas de Probabilidades de Emisión ===")
    suma_correcta = True
    for etiqueta in emision:
        suma = sum(emision[etiqueta].values())
        if not np.isclose(suma, 1.0, rtol=1e-5):
            print(f"Error: Suma de P(palabra|{etiquzeńta}) = {suma:.6f} (debería ser ~1.0)")
            suma_correcta = False
    if suma_correcta:
        print("Suma de probabilidades por etiqueta: Correcta (~1.0 para todas las etiquetas)")

    palabras_corpus = set(token for token, _ in corpus)
    palabras_cubiertas = set()
    for etiqueta in emision:
        palabras_cubiertas.update(emision[etiqueta].keys())
    cobertura = len(palabras_cubiertas) / len(palabras_corpus) * 100
    print(f"Cobertura de palabras: {cobertura:.2f}% ({len(palabras_cubiertas)} de {len(palabras_corpus)} palabras)")

    entropias = []
    for etiqueta in emision:
        probs = np.array(list(emision[etiqueta].values()))
        entropia = -np.sum(probs * np.log2(probs + 1e-10))
        entropias.append(entropia)
    entropia_promedio = np.mean(entropias)
    print(f"Entropía promedio por etiqueta: {entropia_promedio:.4f} bits")

    etiquetas = set(etiqueta for _, etiqueta in corpus)
    total_combinaciones = len(palabras_corpus) * len(etiquetas)
    combinaciones_no_nulas = sum(len(emision[etiqueta]) for etiqueta in emision)
    sparsity = (1 - combinaciones_no_nulas / total_combinaciones) * 100
    print(f"Sparsity: {sparsity:.2f}% ({total_combinaciones - combinaciones_no_nulas} de {total_combinaciones} combinaciones tienen P=0)")

    return emision

def calcular_probabilidades_transicion(oraciones):
    """
    Calcula las probabilidades de transición P(etiqueta_i|etiqueta_{i-1}) con suavizado de Laplace, incluyendo <START> y <END>.
    Args:
        oraciones (list): Lista de tuplas (doc_id, oracion), donde oracion es una lista de tuplas (token, etiqueta).
    Returns:
        dict: Diccionario con etiquetas (incluyendo <START> y <END>) como claves y otro diccionario {etiqueta_siguiente: probabilidad} como valor.
    """
    transicion = {}
    total_transiciones = {}
    etiquetas = set(["<START>", "<END>"]) | set(etiqueta for _, oracion in oraciones for _, etiqueta in oracion)  # Todas las etiquetas + <START> y <END>
    k = 1  # Constante de suavizado de Laplace

    # Inicializar contadores para todas las etiquetas
    for etiqueta in etiquetas:
        transicion[etiqueta] = {sig: 0 for sig in etiquetas}  # Inicializar con 0 para todas las etiquetas siguientes
        total_transiciones[etiqueta] = 0

    # Contar transiciones dentro de cada oración
    for doc_id, oracion in oraciones:
        if not oracion:  # Saltar oraciones vacías
            continue
        anterior = "<START>"
        for _, etiqueta in oracion:
            if anterior not in transicion:  # Asegurar que anterior esté inicializado
                transicion[anterior] = {sig: 0 for sig in etiquetas}
                total_transiciones[anterior] = 0
            transicion[anterior][etiqueta] += 1
            total_transiciones[anterior] += 1
            anterior = etiqueta
            if anterior not in transicion:  # Asegurar que la nueva etiqueta esté inicializada
                transicion[anterior] = {sig: 0 for sig in etiquetas}
                total_transiciones[anterior] = 0
        # Añadir transición al estado final <END>
        transicion[anterior]["<END>"] += 1
        total_transiciones[anterior] += 1
    # Aplicar suavizado de Laplace
    for etiqueta in transicion:
        for siguiente in transicion[etiqueta]:
            # P(etiqueta_i|etiqueta_{i-1}) = (count(etiqueta_{i-1}, etiqueta_i) + k) / (total_count(etiqueta_{i-1}) + k * |etiquetas|)
            transicion[etiqueta][siguiente] = (transicion[etiqueta][siguiente] + k) / (total_transiciones[etiqueta] + k * len(etiquetas))
    # Métricas de efectividad
    print("\n=== Métricas de Probabilidades de Transición ===")
    suma_correcta = True
    for etiqueta in transicion:
        suma = sum(transicion[etiqueta].values())
        if not np.isclose(suma, 1.0, rtol=1e-5):
            print(f"Error: Suma de P(etiqueta_i|{etiqueta}) = {suma:.6f} (debería ser ~1.0)")
            suma_correcta = False
    if suma_correcta:
        print("Suma de probabilidades por etiqueta anterior: Correcta (~1.0 para todas las etiquetas)")
    total_transiciones_posibles = len(etiquetas) * len(etiquetas)
    transiciones_no_nulas = sum(sum(1 for sig in transicion[etiqueta] if transicion[etiqueta][sig] > k / (total_transiciones[etiqueta] + k * len(etiquetas))) for etiqueta in transicion)
    cobertura = (transiciones_no_nulas / total_transiciones_posibles) * 100
    print(f"Cobertura de transiciones: {cobertura:.2f}% ({transiciones_no_nulas} de {total_transiciones_posibles} transiciones)")
    entropias = []
    for etiqueta in transicion:
        probs = np.array(list(transicion[etiqueta].values()))
        entropia = -np.sum(probs * np.log2(probs + 1e-10))
        entropias.append(entropia)
    entropia_promedio = np.mean(entropias)
    print(f"Entropía promedio por etiqueta anterior: {entropia_promedio:.4f} bits")
    sparsity = (1 - transiciones_no_nulas / total_transiciones_posibles) * 100
    print(f"Sparsity: {sparsity:.2f}% ({total_transiciones_posibles - transiciones_no_nulas} de {total_transiciones_posibles} transiciones tienen P=0)")
    return transicion

# Calcular probabilidades con suavizado
emision = calcular_probabilidades_emision(corpus_flat)
transicion = calcular_probabilidades_transicion(oraciones)


=== Punto 3: Implementación de Calcular Probabilidades con Suavizado de Laplace ===


=== Métricas de Probabilidades de Emisión ===
Suma de probabilidades por etiqueta: Correcta (~1.0 para todas las etiquetas)
Cobertura de palabras: 100.00% (91 de 91 palabras)
Entropía promedio por etiqueta: 6.4595 bits
Sparsity: 0.00% (0 de 2639 combinaciones tienen P=0)

=== Métricas de Probabilidades de Transición ===
Suma de probabilidades por etiqueta anterior: Correcta (~1.0 para todas las etiquetas)
Cobertura de transiciones: 7.91% (76 de 961 transiciones)
Entropía promedio por etiqueta anterior: 4.8188 bits
Sparsity: 92.09% (885 de 961 transiciones tienen P=0)


In [78]:
print("\n=== Punto 4: Implementar el Algoritmo de Viterbi ===\n")

def viterbi(frase, estados, transicion, emision):
    """
    Implementa el algoritmo de Viterbi para encontrar la secuencia de etiquetas más probable.
    Incluye transición al estado final <END>.
    Args:
        frase (list): Lista de palabras de la oración.
        estados (list): Lista de etiquetas posibles (sin incluir <START> ni <END>).
        transicion (dict): Probabilidades de transición P(etiqueta_i|etiqueta_{i-1}).
        emision (dict): Probabilidades de emisión P(palabra|etiqueta).
    Returns:
        tuple: (matriz de probabilidades, probabilidad total, lista de etiquetas de la ruta más probable).
    """
    V = [{}]  # Matriz de probabilidades de Viterbi
    path = {}  # Rutas para cada estado
    matriz = [{} for _ in range(len(frase))]  # Almacena la matriz de probabilidades para exportar

    # Inicialización: transición desde <START> a la primera etiqueta
    for estado in estados:
        prob = transicion["<START>"][estado] * emision[estado].get(frase[0], 1e-6)  # Usar 1e-6 para palabras no vistas
        V[0][estado] = prob
        matriz[0][estado] = prob
        path[estado] = [estado]

    # Propagación: calcular probabilidades para cada palabra
    for t in range(1, len(frase)):
        V.append({})
        new_path = {}
        for y in estados:
            # Máxima probabilidad considerando la transición desde el estado anterior
            (prob, estado_max) = max(
                [(V[t-1][y0] * transicion[y0][y] * emision[y].get(frase[t], 1e-6), y0)
                 for y0 in estados], key=lambda x: x[0])
            V[t][y] = prob
            matriz[t][y] = prob
            new_path[y] = path[estado_max] + [y]
        path = new_path

    # Finalización: transición al estado final <END>
    n = len(frase) - 1
    V.append({})
    for y in estados:
        V[n+1]["<END>"] = V[n][y] * transicion[y]["<END>"]
    (prob, estado_max) = max([(V[n][y] * transicion[y]["<END>"], y) for y in estados], key=lambda x: x[0])
    ruta = path[estado_max]  # La ruta no incluye <END>, ya que solo se piden las etiquetas de las palabras

    return matriz, prob, ruta


=== Punto 4: Implementar el Algoritmo de Viterbi ===



In [79]:
print("\n=== Punto 5: Etiquetar las Oraciones ===\n")

# Etiquetar la primera oración
frase_prueba = ['Habla', 'con', 'el', 'enfermo', 'grave', 'de', 'trasplantes', '.']
# Filtrar estados relevantes para la oración
estados_relevantes = set()
for token in frase_prueba:
    for etiqueta in emision:
        if token in emision[etiqueta] and emision[etiqueta][token] > 1e-6:  # Umbral para evitar ruido del suavizado
            estados_relevantes.add(etiqueta)
estados = list(estados_relevantes)

matriz, probabilidad, ruta = viterbi(frase_prueba, estados, transicion, emision)
print("Frase:", frase_prueba)
print("Ruta más probable:", ruta)
print("Probabilidad total:", probabilidad)

# Validar contra el corpus (doc id=1)
ref_oracion1 = [('Habla', 'VMIP3S0'), ('con', 'SP'), ('el', 'DA'), ('enfermo', 'NCMS000'), ('grave', 'AQ0CS00'), ('de', 'SP'), ('trasplantes', 'NCMN000'), ('.', 'Fp')]
print("Validación Oración 1:", "Correcta" if list(zip(frase_prueba, ruta)) == ref_oracion1 else "Incorrecta")

# Etiquetar la segunda oración
frase_prueba2 = ['El', 'enfermo', 'grave', 'habla', 'de', 'trasplantes', '.']
# Filtrar estados relevantes para la oración
estados_relevantes2 = set()
for token in frase_prueba2:
    for etiqueta in emision:
        if token in emision[etiqueta] and emision[etiqueta][token] > 1e-6:  # Umbral para evitar ruido
            estados_relevantes2.add(etiqueta)
estados2 = list(estados_relevantes2)

matriz2, probabilidad2, ruta2 = viterbi(frase_prueba2, estados2, transicion, emision)
print("\nFrase 2:", frase_prueba2)
print("Ruta más probable 2:", ruta2)
print("Probabilidad total 2:", probabilidad2)

# Validar contra el corpus (doc id=2)
ref_oracion2 = [('El', 'DA'), ('enfermo', 'NCMS000'), ('grave', 'AQ0CS00'), ('habla', 'VMIP3S0'), ('de', 'SP'), ('trasplantes', 'NCMN000'), ('.', 'Fp')]
print("Validación Oración 2:", "Correcta" if list(zip(frase_prueba2, ruta2)) == ref_oracion2 else "Incorrecta")


=== Punto 5: Etiquetar las Oraciones ===

Frase: ['Habla', 'con', 'el', 'enfermo', 'grave', 'de', 'trasplantes', '.']
Ruta más probable: ['VMIP3S0', 'SP', 'DA', 'NCMS000', 'AQ0CS00', 'SP', 'NCMN000', 'Fp']
Probabilidad total: 6.96740681967439e-21
Validación Oración 1: Correcta

Frase 2: ['El', 'enfermo', 'grave', 'habla', 'de', 'trasplantes', '.']
Ruta más probable 2: ['DA', 'NCMS000', 'AQ0CS00', 'VMIP3S0', 'SP', 'NCMN000', 'Fp']
Probabilidad total 2: 8.16166389485921e-18
Validación Oración 2: Correcta


In [80]:
print("\n=== Punto 6: Generar Tablas en Archivos Excel ===\n")

!pip install -q openpyxl
import pandas as pd

try:
    # Generar tablas de emisión y transición
    df_emision = pd.DataFrame.from_dict(emision, orient='index').fillna(0)
    df_transicion = pd.DataFrame.from_dict(transicion, orient='index').fillna(0)

    # Generar tabla de probabilidades iniciales desde <START>
    df_iniciales = pd.DataFrame({
        "Etiqueta": list(transicion["<START>"].keys()),
        "Probabilidad": list(transicion["<START>"].values())
    })

    # Generar matriz de Viterbi para la primera oración
    df_viterbi = pd.DataFrame(matriz, index=frase_prueba, columns=estados).T.fillna(0)

    # Generar tabla de rutas para la primera oración
    df_ruta_viterbi = pd.DataFrame({
        "Palabra": frase_prueba,
        "Etiqueta": ruta
    })

    # Generar matriz de Viterbi para la segunda oración
    df_viterbi2 = pd.DataFrame(matriz2, index=frase_prueba2, columns=estados2).T.fillna(0)

    # Generar tabla de rutas para la segunda oración
    df_ruta_viterbi2 = pd.DataFrame({
        "Palabra": frase_prueba2,
        "Etiqueta": ruta2
    })

    # Guardar en Excel
    df_emision.to_excel("tabla_emision.xlsx", engine='openpyxl')
    df_transicion.to_excel("tabla_transicion.xlsx", engine='openpyxl')
    df_iniciales.to_excel("tabla_iniciales.xlsx", engine='openpyxl')
    df_viterbi.to_excel("ruta_viterbi.xlsx", engine='openpyxl')
    df_ruta_viterbi.to_excel("etiquetas_viterbi.xlsx", engine='openpyxl')
    df_viterbi2.to_excel("matriz_viterbi_sentence2.xlsx", engine='openpyxl')
    df_ruta_viterbi2.to_excel("etiquetas_viterbi_sentence2.xlsx", engine='openpyxl')

    # Descargar archivos
    from google.colab import files
    files.download("tabla_emision.xlsx")
    files.download("tabla_transicion.xlsx")
    files.download("tabla_iniciales.xlsx")
    files.download("ruta_viterbi.xlsx")
    files.download("etiquetas_viterbi.xlsx")
    files.download("matriz_viterbi_sentence2.xlsx")
    files.download("etiquetas_viterbi_sentence2.xlsx")

except NameError as e:
    print(f"Error: {e}. Asegúrate de ejecutar las celdas 1 a 5 en orden antes de esta celda.")


=== Punto 6: Generar Tablas en Archivos Excel ===



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [81]:
print("\n=== Punto 7: Preguntas de Análisis ===\n")

import pandas as pd

print("\n1. ¿Es correcto el etiquetado morfosintáctico de la primera oración?")
print(f"La oración etiquetada es: {list(zip(frase_prueba, ruta))}")
print("""
El etiquetado es correcto, como se valida al compararlo con el corpus (doc id=1):
- Habla: VMIP3S0 (verbo principal, indicativo, presente, tercera persona singular).
- con: SP (preposición).
- el: DA (artículo definido).
- enfermo: NCMS000 (nombre común, masculino, singular).
- grave: AQ0CS00 (adjetivo calificativo, común, singular).
- de: SP (preposición).
- trasplantes: NCMN000 (nombre común, masculino, sin especificación de número).
- .: Fp (puntuación).
El suavizado de Laplace asegura probabilidades no nulas para palabras no vistas, el estado <START> modela correctamente las transiciones iniciales (ver tabla_iniciales.xlsx), y el estado <END> captura la transición final. La matriz de Viterbi (ruta_viterbi.xlsx) muestra las probabilidades calculadas, y la validación explícita confirma la corrección.
""")

print("\n2. Resultado del etiquetado de la segunda oración y ¿por qué es correcto?")
print(f"La oración etiquetada es: {list(zip(frase_prueba2, ruta2))}")
print("""
El etiquetado es correcto, como se valida al compararlo con el corpus (doc id=2):
- El: DA (artículo definido).
- enfermo: NCMS000 (nombre común, masculino, singular).
- grave: AQ0CS00 (adjetivo calificativo, común, singular).
- habla: VMIP3S0 (verbo principal, indicativo, presente, tercera persona singular).
- de: SP (preposición).
- trasplantes: NCMN000 (nombre común, masculino, sin especificación de número).
- .: Fp (puntuación).
La matriz de Viterbi (matriz_viterbi_sentence2.xlsx) confirma las probabilidades. El modelo captura la estructura diferente gracias al suavizado, el estado <START> para transiciones iniciales, y <END> para la transición final (ver tabla_iniciales.xlsx y tabla_transicion.xlsx).
""")

print("\n3. ¿Cuáles son las limitaciones del etiquetador morfosintáctico creado?")
print("""
- **Corpus pequeño**: Con solo 20 oraciones y 137 tokens, el modelo tiene una representatividad limitada, lo que puede causar sobreajuste y poca generalización.
- **Bigramas**: El modelo HMM de bigramas solo considera la etiqueta anterior, ignorando contextos más largos que podrían resolver ambigüedades.
- **Suavizado de Laplace**: Aunque robusto, asigna probabilidades uniformes a eventos no vistos, lo que puede ser subóptimo para palabras o transiciones raras.
- **Ambigüedad léxica**: Palabras como 'habla' (verbo o sustantivo) pueden etiquetarse incorrectamente si el contexto no es suficiente.
- **Dependencia del corpus**: La calidad del etiquetado depende de la precisión del corpus anotado (por ejemplo, FreeLing).
""")

print("\n4. ¿Qué posibles mejoras se podrían aplicar?")
print("""
- **Corpus más grande**: Usar un corpus más extenso (por ejemplo, todo el Wikicorpus) para mejorar la diversidad de palabras y transiciones.
- **Modelos de trigramas**: Incorporar HMM de orden superior para capturar contextos más largos.
- **Suavizado avanzado**: Aplicar suavizado Kneser-Ney para una mejor distribución de probabilidades en eventos raros.
- **Ingeniería de características**: Usar lemas, sufijos o información morfológica para manejar palabras no vistas.
- **Modelos híbridos**: Combinar HMM con reglas lingüísticas o modelos neuronales (por ejemplo, transformers) para mejorar la precisión.
- **Evaluación en conjunto de prueba**: Validar el modelo en un conjunto de datos independiente para medir su rendimiento.
""")

print("\n5. Análisis de las Probabilidades Iniciales (tabla_iniciales.xlsx)")
try:
    # Leer tabla_iniciales.xlsx
    df_iniciales = pd.read_excel("tabla_iniciales.xlsx")

    # Ordenar por probabilidad descendente y seleccionar las 5 etiquetas más probables
    top_5_iniciales = df_iniciales.sort_values(by="Probabilidad", ascending=False).head(5)

    print("\nLas 5 etiquetas iniciales más probables:")
    for idx, row in top_5_iniciales.iterrows():
        print(f"- {row['Etiqueta']}: {row['Probabilidad']:.4f}")

    print("""
Observaciones:
- Las etiquetas iniciales más probables reflejan estructuras comunes en el corpus:
  - Artículos definidos (DA) son frecuentes al inicio de frases nominales, como en la Oración 2 ("El enfermo...").
  - Verbos en presente (VMIP3S0) son comunes en oraciones narrativas, como en la Oración 1 ("Habla...").
- El suavizado de Laplace asigna probabilidades no nulas a etiquetas no vistas como iniciales, asegurando robustez.
- La inclusión de <START> y <END> modela correctamente los límites de las oraciones, mejorando las transiciones iniciales y finales.
- Un corpus más grande mejoraría la precisión de estas probabilidades, ya que 20 oraciones limitan la representatividad.
""")
except FileNotFoundError:
    print("Error: No se encontró tabla_iniciales.xlsx. Asegúrate de ejecutar la celda 6 primero.")


=== Punto 7: Preguntas de Análisis ===


1. ¿Es correcto el etiquetado morfosintáctico de la primera oración?
La oración etiquetada es: [('Habla', 'VMIP3S0'), ('con', 'SP'), ('el', 'DA'), ('enfermo', 'NCMS000'), ('grave', 'AQ0CS00'), ('de', 'SP'), ('trasplantes', 'NCMN000'), ('.', 'Fp')]

El etiquetado es correcto, como se valida al compararlo con el corpus (doc id=1):
- Habla: VMIP3S0 (verbo principal, indicativo, presente, tercera persona singular).
- con: SP (preposición).
- el: DA (artículo definido).
- enfermo: NCMS000 (nombre común, masculino, singular).
- grave: AQ0CS00 (adjetivo calificativo, común, singular).
- de: SP (preposición).
- trasplantes: NCMN000 (nombre común, masculino, sin especificación de número).
- .: Fp (puntuación).
El suavizado de Laplace asegura probabilidades no nulas para palabras no vistas, el estado <START> modela correctamente las transiciones iniciales (ver tabla_iniciales.xlsx), y el estado <END> captura la transición final. La matriz de Viterbi (rut